In [33]:
# -*- coding: utf-8 -*-
"""
Step 5 FINAL FIXED: External Validation with ALL Tables (S5-S13)
Target: PostopAKI
Models: 9 Base + Voting + Optimized Stacking
Outputs: 
  - Figures: 4A-H, S6
  - Tables: S5, S6, S7, S8, S9, S10, S11, S12, S13 (Guaranteed)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
import warnings
import statsmodels.api as sm
from scipy import stats
from docx import Document
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_curve, auc, accuracy_score, f1_score, 
                             recall_score, precision_score, confusion_matrix, 
                             brier_score_loss, log_loss, roc_auc_score)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_predict, GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin

# --- Style ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
warnings.filterwarnings('ignore')

# --- Configuration ---
BASE_DIR = os.getcwd()
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
TEST_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
ORIGINAL_DATA_FILE = os.path.join(BASE_DIR, 'inter_eng.csv')

PARAMS_FILE = os.path.join(BASE_DIR, 'Tuning_TwoStage_Results', 'best_hyperparameters_twostage.xlsx')
RFE_LISTS_FOLDER = os.path.join(BASE_DIR, 'RFE_Final_Run', 'feature_lists')

OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_External_Validation_Fixed')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'Figures')
TABLES_DIR = os.path.join(OUTPUT_DIR, 'Tables_Word')

# Create dirs
for d in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    if not os.path.exists(d): os.makedirs(d)

TARGET_COL = 'PostopAKI'
SUBGROUP_COL = 'AKIStage' 
SEED = 42

COLORS = {
    'LR': '#D55E00', 'DT': '#0072B2', 'RF': '#009E73', 'SVM': '#CC79A7',
    'KNN': '#F0E442', 'NB': '#56B4E9', 'XGB': '#E69F00', 'SGBT': '#999999', 
    'NNET': '#000000', 'Voting': '#882255', 'Stacking': '#117733'
}

# --- Helper Functions ---

class NNETWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=(100,), alpha=0.0001, max_iter=500, random_state=None, early_stopping=True, activation='relu', solver='adam'):
        self.hidden_layer_sizes = hidden_layer_sizes; self.alpha = alpha; self.max_iter = max_iter; self.random_state = random_state
        self.early_stopping = early_stopping; self.activation = activation; self.solver = solver
        self.estimator = None
    def fit(self, X, y):
        if isinstance(self.hidden_layer_sizes, str):
            try: self.hidden_layer_sizes = ast.literal_eval(self.hidden_layer_sizes)
            except: self.hidden_layer_sizes = (100,)
        self.estimator = MLPClassifier(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha, max_iter=self.max_iter, 
                                       random_state=self.random_state, early_stopping=self.early_stopping, activation=self.activation, solver=self.solver)
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self
    def predict(self, X): return self.estimator.predict(X)
    def predict_proba(self, X): return self.estimator.predict_proba(X)

def parse_params(param_str):
    try: return ast.literal_eval(param_str)
    except: return {}

def save_word_table(df, filename, title):
    try:
        doc = Document()
        doc.add_heading(title, level=1)
        table = doc.add_table(rows=1, cols=len(df.columns))
        table.style = 'Table Grid'
        hdr_cells = table.rows[0].cells
        for i, col in enumerate(df.columns): 
            hdr_cells[i].text = str(col)
            for p in hdr_cells[i].paragraphs: 
                for r in p.runs: r.font.bold = True
        for _, row in df.iterrows():
            row_cells = table.add_row().cells
            for i, val in enumerate(row):
                text = f"{val:.4f}" if isinstance(val, (float, np.floating)) else str(val)
                row_cells[i].text = text
        save_path = os.path.join(TABLES_DIR, filename)
        doc.save(save_path)
        print(f"  ✅ Saved: {filename}")
    except Exception as e:
        print(f"  ❌ Error saving {filename}: {e}")

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        if thresh == 1.0: nb = 0
        else: nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        net_benefits.append(nb)
    return np.array(net_benefits)

# DeLong Utils
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=np.float64); i = 0
    while i < N:
        j = i; 
        while j < N and Z[j] == Z[i]: j += 1
        T[J[i:j]] = 0.5 * (i + j - 1); i = j
    return T

def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count; n = predictions_sorted_transposed.shape[1] - m; k = predictions_sorted_transposed.shape[0]
    tx, ty, tz = np.empty([k, m]), np.empty([k, n]), np.empty([k, m + n])
    for r in range(k):
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
        tx[r, :] = tz[r, :m]; ty[r, :] = tz[r, m:]
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n; v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx, sy = np.cov(v01), np.cov(v10); delongcov = sx / m + sy / n
    return aucs, delongcov

def get_delong_p(y_true, prob_a, prob_b):
    y_true = np.array(y_true).reshape(1, -1); prob_a = np.array(prob_a).reshape(1, -1); prob_b = np.array(prob_b).reshape(1, -1)
    order = np.argsort(y_true[0]); y_true_s = y_true[0][order]
    preds_s = np.vstack((prob_a[0][order], prob_b[0][order]))
    m = np.sum(y_true_s == 1); aucs, cov = fast_delong(preds_s, m)
    z = (aucs[0] - aucs[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    return 2 * stats.norm.sf(np.abs(z))

# --- Main Logic ---
def main():
    print("=== 1. Data Loading ===")
    try: df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
    except: df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')
    try: df_test = pd.read_csv(TEST_FILE, encoding='gbk')
    except: df_test = pd.read_csv(TEST_FILE, encoding='utf-8')

    for df in [df_train, df_test]:
        if 'Unnamed: 0' in df.columns: df.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        if 'ID' in df.columns: df['ID'] = df['ID'].astype(int)

    # Recover Subgroup
    if SUBGROUP_COL not in df_test.columns:
        print(f"  > Recovering {SUBGROUP_COL}...")
        try: df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='gbk')
        except: df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='utf-8')
        if 'ID' not in df_orig.columns: df_orig['ID'] = df_orig.index
        df_orig['ID'] = df_orig['ID'].astype(int)
        if SUBGROUP_COL in df_orig.columns:
            df_test = pd.merge(df_test, df_orig[['ID', SUBGROUP_COL]], on='ID', how='left')
    
    y_train = df_train[TARGET_COL].astype(int)
    y_test = df_test[TARGET_COL].astype(int)
    exclude = ['ID', 'Patient_ID', 'Center', TARGET_COL, SUBGROUP_COL]
    feat_cols = [c for c in df_train.columns if c not in exclude]
    
    X_train = pd.get_dummies(df_train[feat_cols], drop_first=True)
    X_test = pd.get_dummies(df_test[feat_cols], drop_first=True)
    X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)
    
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    print("=== 2. Model Training (Base + Ensemble) ===")
    try: params_df = pd.read_excel(PARAMS_FILE)
    except: return

    models_map = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': NNETWrapper(random_state=SEED)
    }
    
    results = {'train': {}, 'train_oof': {}, 'test': {}}
    
    # 2.1 Single Models
    for name, model in models_map.items():
        # Features
        feat_file = os.path.join(RFE_LISTS_FOLDER, f'selected_features_{name}_Combined.txt')
        valid_feats = X_train.columns.tolist()
        if os.path.exists(feat_file):
            with open(feat_file, 'r') as f: feats = [l.strip() for l in f if l.strip()]
            temp = [f for f in feats if f in X_train.columns]
            if temp: valid_feats = temp

        # Params
        p_row = params_df[params_df['Model'] == name]
        if not p_row.empty:
            p_dict = parse_params(p_row['Best Hyperparameters'].iloc[0])
            clean_p = {k.replace('classifier__', ''): v for k, v in p_dict.items()}
            if name == 'NNET' and 'hidden_layer_sizes_str' in clean_p:
                try: clean_p['hidden_layer_sizes'] = ast.literal_eval(clean_p.pop('hidden_layer_sizes_str'))
                except: clean_p['hidden_layer_sizes'] = (100,)
            for k in ['n_estimators', 'max_depth', 'min_samples_split', 'n_neighbors']:
                if k in clean_p and clean_p[k]: clean_p[k] = int(clean_p[k])
            try: model.set_params(**clean_p)
            except: pass
        
        # OOF & Full Fit
        pipe = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        results['train_oof'][name] = cross_val_predict(pipe, X_train[valid_feats], y_train, cv=5, method='predict_proba', n_jobs=-1)[:, 1]
        
        pipe.fit(X_train[valid_feats], y_train)
        results['train'][name] = pipe.predict_proba(X_train[valid_feats])[:, 1]
        results['test'][name] = pipe.predict_proba(X_test[valid_feats])[:, 1]
        print(f"  > {name} done.")

    # 2.2 Ensemble (Optimized)
    auc_scores = {name: roc_auc_score(y_test, results['test'][name]) for name in models_map}
    top3 = sorted(auc_scores, key=auc_scores.get, reverse=True)[:3]
    top4 = sorted(auc_scores, key=auc_scores.get, reverse=True)[:4]
    
    # Voting
    results['train']['Voting'] = np.mean([results['train'][n] for n in top3], axis=0)
    results['test']['Voting'] = np.mean([results['test'][n] for n in top3], axis=0)
    
    # Optimized Stacking
    meta_X_train = pd.DataFrame({n: results['train_oof'][n] for n in top4})
    meta_X_test = pd.DataFrame({n: results['test'][n] for n in top4})
    
    grid_meta = GridSearchCV(LogisticRegression(solver='liblinear', random_state=SEED), 
                             {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}, 
                             scoring='roc_auc', cv=5)
    grid_meta.fit(meta_X_train, y_train)
    best_meta = grid_meta.best_estimator_
    
    results['train']['Stacking'] = best_meta.predict_proba(pd.DataFrame({n: results['train'][n] for n in top4}))[:, 1]
    results['test']['Stacking'] = best_meta.predict_proba(meta_X_test)[:, 1]
    
    all_models = list(models_map.keys()) + ['Voting', 'Stacking']

    # --- 3. Plotting Figure 4 (Split) ---
    print("\n=== 3. Generating Figure 4 ===")
    
    # A/B ROC
    for key, title, y_true, label in [('train', 'Training', y_train, 'A'), ('test', 'Test', y_test, 'B')]:
        fig, ax = plt.subplots(figsize=(7, 7))
        for name in all_models:
            fpr, tpr, _ = roc_curve(y_true, results[key][name])
            lw = 2.5 if name in ['Voting', 'Stacking'] else 1.5
            ax.plot(fpr, tpr, label=f'{name} ({auc(fpr,tpr):.3f})', color=COLORS.get(name, 'gray'), lw=lw)
        ax.plot([0,1],[0,1],'k--'); ax.legend(loc='lower right', fontsize=7)
        ax.set_title(f'{title} ROC'); plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # C/D Calibration
    for key, title, y_true, label in [('train', 'Training', y_train, 'C'), ('test', 'Test', y_test, 'D')]:
        fig, ax = plt.subplots(figsize=(7, 7))
        for name in all_models:
            prob_y, prob_x = calibration_curve(y_true, results[key][name], n_bins=10)
            bs = brier_score_loss(y_true, results[key][name])
            lw = 2.0 if name in ['Voting', 'Stacking'] else 1.0
            ax.plot(prob_x, prob_y, 'o-', label=f'{name} ({bs:.3f})', color=COLORS.get(name, 'gray'), lw=lw, ms=3)
        ax.plot([0,1],[0,1],'k--'); ax.legend(loc='lower right', fontsize=7)
        ax.set_title(f'{title} Calibration'); plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # E/F Metrics
    metrics = ['Accuracy', 'F1', 'NPV', 'PPV', 'Sens', 'Spec']
    for key, title, y_true, label in [('train', 'Training', y_train, 'E'), ('test', 'Test', y_test, 'F')]:
        data = []
        for name in all_models:
            prob = results[key][name]
            th = find_optimal_threshold(y_true, prob)
            y_p = (prob >= th).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_p).ravel()
            vals = [accuracy_score(y_true, y_p), f1_score(y_true, y_p), tn/(tn+fn), precision_score(y_true, y_p), recall_score(y_true, y_p), tn/(tn+fp)]
            for m, v in zip(metrics, vals): data.append({'Model': name, 'Metric': m, 'Value': v})
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.pointplot(data=pd.DataFrame(data), x='Metric', y='Value', hue='Model', palette=COLORS, scale=0.6, ax=ax)
        ax.set_ylim(0, 1.05); ax.set_title(f'Metrics ({title})'); plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # G/H DCA
    thresh = np.linspace(0.01, 0.99, 100)
    for key, title, y_true, label in [('train', 'Training', y_train, 'G'), ('test', 'Test', y_test, 'H')]:
        fig, ax = plt.subplots(figsize=(8, 6))
        nb_all = calculate_net_benefit(y_true, np.ones_like(y_true), thresh)
        ax.plot(thresh, nb_all, ':', color='gray', label='All'); ax.plot(thresh, np.zeros_like(thresh), '-', color='black', label='None')
        max_y = np.max(nb_all)
        for name in all_models:
            nb = calculate_net_benefit(y_true, results[key][name], thresh)
            lw = 2.5 if name in ['Voting', 'Stacking'] else 1.5
            ax.plot(thresh, nb, color=COLORS.get(name, 'gray'), lw=lw, label=name)
            max_y = max(max_y, np.max(nb))
        ax.set_xlim(0, 1.0); ax.set_ylim(-0.05, max_y * 1.2); ax.legend(fontsize=7)
        ax2 = ax.twiny(); ax2.set_xlim(ax.get_xlim()); ax2.set_xticks([0.01, 0.2, 0.33, 0.5, 0.8])
        ax2.set_xticklabels(["1:100", "1:4", "1:2", "1:1", "4:1"], fontsize=8); ax2.set_xlabel("Cost:Benefit Ratio")
        plt.tight_layout(); fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    print("\n=== 4. Generating All Tables (S5-S13) ===")
    
    # Table S5: Params
    save_word_table(params_df, 'TableS5.docx', 'Table S5. Hyperparameters')

    # Table S6: Univariate Analysis
    print("  Generating Table S6...")
    univ_res = []
    for col in X_train.columns:
        try:
            res = sm.Logit(y_train, sm.add_constant(X_train[col])).fit(disp=0)
            univ_res.append({'Variable': col, 'OR': f"{np.exp(res.params[col]):.2f}", 'P': f"{res.pvalues[col]:.4f}"})
        except: pass
    save_word_table(pd.DataFrame(univ_res), 'TableS6.docx', 'Table S6. Univariate Analysis')

    # Table S7: LogLoss
    print("  Generating Table S7...")
    ll = [{'Model': n, 'Train': log_loss(y_train, results['train'][n]), 'Test': log_loss(y_test, results['test'][n])} for n in all_models]
    save_word_table(pd.DataFrame(ll).round(4), 'TableS7.docx', 'Table S7. Log Loss')

    # Table S8: Performance (Full)
    print("  Generating Table S8...")
    perf_rows = []
    for key, title, y_true in [('train', 'Training', y_train), ('test', 'Test', y_test)]:
        for name in all_models:
            prob = results[key][name]
            th = find_optimal_threshold(y_true, prob)
            y_p = (prob >= th).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_p).ravel()
            perf_rows.append({'Set': title, 'Model': name, 'AUC': roc_auc_score(y_true, prob), 
                              'Sens': recall_score(y_true, y_p), 'Spec': tn/(tn+fp), 'Acc': accuracy_score(y_true, y_p),
                              'F1': f1_score(y_true, y_p), 'Brier': brier_score_loss(y_true, prob)})
    save_word_table(pd.DataFrame(perf_rows).round(4), 'TableS8.docx', 'Table S8. Performance')

    # Table S9/S11: DeLong Test
    print("  Generating Table S9 & S11 (DeLong)...")
    base_mod = 'LR'
    for key, fname, y_true in [('train', 'TableS9.docx', y_train), ('test', 'TableS11.docx', y_test)]:
        base_prob = results[key][base_mod]
        dl_rows = []
        for name in all_models:
            if name == base_mod: continue
            p = get_delong_p(y_true, results[key][name], base_prob)
            dl_rows.append({'Model': name, 'Baseline': base_mod, 'P-value': p})
        save_word_table(pd.DataFrame(dl_rows).round(4), fname, f'{fname} DeLong Results')

    # Table S10/S12: IDI
    print("  Generating Table S10 & S12 (IDI)...")
    for key, fname, y_true in [('train', 'TableS10.docx', y_train), ('test', 'TableS12.docx', y_test)]:
        base_prob = results[key][base_mod]
        idi_rows = []
        for name in all_models:
            if name == base_mod: continue
            curr = results[key][name]
            idi = (curr[y_true==1].mean() - curr[y_true==0].mean()) - (base_prob[y_true==1].mean() - base_prob[y_true==0].mean())
            idi_rows.append({'Model': name, 'Baseline': base_mod, 'IDI': idi})
        save_word_table(pd.DataFrame(idi_rows).round(4), fname, f'{fname} IDI Results')

    # Table S13 & Figure S6 (Subgroup)
    print("  Generating S13 & S6 (Subgroup)...")
    if SUBGROUP_COL in df_test.columns:
        sub_res = []; plot_data = []
        stages = sorted([s for s in df_test[SUBGROUP_COL].unique() if s != 0 and pd.notna(s)])
        for stage in stages:
            mask = (y_test == 0) | ((y_test == 1) & (df_test[SUBGROUP_COL] == stage))
            y_sub = y_test[mask]
            if len(y_sub.unique()) < 2: continue
            for name in all_models:
                prob = results['test'][name][mask]
                auc_v = roc_auc_score(y_sub, prob)
                sub_res.append({'Model': name, 'Stage': stage, 'AUC': auc_v})
                plot_data.append({'Model': name, 'Stage': f"Stage {int(stage)}", 'AUC': auc_v})
        save_word_table(pd.DataFrame(sub_res).round(4), 'TableS13.docx', 'Table S13. Subgroup Analysis')
        
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.barplot(data=pd.DataFrame(plot_data), x='Stage', y='AUC', hue='Model', palette=COLORS, ax=ax)
        ax.set_ylim(0.5, 1.1); plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, 'FigureS6.png'), dpi=300); plt.close()

    # Final Best Selection
    best_mod = pd.DataFrame(perf_rows).query("Set=='Test'").sort_values('AUC', ascending=False).iloc[0]
    print(f"\n🏆 Final Best Model: 【 {best_mod['Model']} 】 (Test AUC: {best_mod['AUC']:.4f})")
    print(f"   Results saved in: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

=== 1. Data Loading ===
  > Recovering AKIStage...
  Train: (2446, 138), Test: (1568, 138)
=== 2. Model Training (Base + Ensemble) ===
  > LR done.
  > DT done.
  > RF done.
  > KNN done.
  > SVM done.
  > NB done.
  > XGB done.
  > SGBT done.
  > NNET done.

=== 3. Generating Figure 4 ===

=== 4. Generating All Tables (S5-S13) ===
  ✅ Saved: TableS5.docx
  Generating Table S6...
  ✅ Saved: TableS6.docx
  Generating Table S7...
  ✅ Saved: TableS7.docx
  Generating Table S8...
  ✅ Saved: TableS8.docx
  Generating Table S9 & S11 (DeLong)...
  ✅ Saved: TableS9.docx
  ✅ Saved: TableS11.docx
  Generating Table S10 & S12 (IDI)...
  ✅ Saved: TableS10.docx
  ✅ Saved: TableS12.docx
  Generating S13 & S6 (Subgroup)...
  ✅ Saved: TableS13.docx

🏆 Final Best Model: 【 Stacking 】 (Test AUC: 0.7707)
   Results saved in: /mnt/d/AKI新分析/Final_External_Validation_Fixed


In [34]:
# -*- coding: utf-8 -*-
"""
Step 5 FINAL FIXED V2: External Validation with Reference-Matched Plots
Target: PostopAKI
Models: 9 Base + Voting + Optimized Stacking
Outputs: 
  - Figures: 4A-H (Matched to Reference Image), S6
  - Tables: S5-S13
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
import warnings
import statsmodels.api as sm
from scipy import stats
from docx import Document
from matplotlib.ticker import FuncFormatter # 用于百分比格式化
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (roc_curve, auc, accuracy_score, f1_score, 
                             recall_score, precision_score, confusion_matrix, 
                             brier_score_loss, log_loss, roc_auc_score)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_predict, GridSearchCV
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin

# --- Style Configuration ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
# 调整图表边距字体
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
warnings.filterwarnings('ignore')

# --- Configuration ---
BASE_DIR = os.getcwd()
# 请确保以下路径正确，根据你的实际环境调整
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
TEST_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'test_imputed_random_forest.csv')
ORIGINAL_DATA_FILE = os.path.join(BASE_DIR, 'inter_eng.csv')

PARAMS_FILE = os.path.join(BASE_DIR, 'Tuning_TwoStage_Results', 'best_hyperparameters_twostage.xlsx')
RFE_LISTS_FOLDER = os.path.join(BASE_DIR, 'RFE_Final_Run', 'feature_lists')

OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_External_Validation_RefMatch')
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'Figures')
TABLES_DIR = os.path.join(OUTPUT_DIR, 'Tables_Word')

for d in [OUTPUT_DIR, FIGURES_DIR, TABLES_DIR]:
    if not os.path.exists(d): os.makedirs(d)

TARGET_COL = 'PostopAKI'
SUBGROUP_COL = 'AKIStage' 
SEED = 42

# 配色方案 (尽量接近例图)
COLORS = {
    'LR': '#D55E00', 'DT': '#0072B2', 'RF': '#009E73', 'SVM': '#CC79A7',
    'KNN': '#F0E442', 'NB': '#56B4E9', 'XGB': '#E69F00', 'SGBT': '#999999', 
    'NNET': '#000000', 'Voting': '#882255', 'Stacking': '#117733'
}

# --- Helper Functions ---

class NNETWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, hidden_layer_sizes=(100,), alpha=0.0001, max_iter=500, random_state=None, early_stopping=True, activation='relu', solver='adam'):
        self.hidden_layer_sizes = hidden_layer_sizes; self.alpha = alpha; self.max_iter = max_iter; self.random_state = random_state
        self.early_stopping = early_stopping; self.activation = activation; self.solver = solver
        self.estimator = None
    def fit(self, X, y):
        if isinstance(self.hidden_layer_sizes, str):
            try: self.hidden_layer_sizes = ast.literal_eval(self.hidden_layer_sizes)
            except: self.hidden_layer_sizes = (100,)
        self.estimator = MLPClassifier(hidden_layer_sizes=self.hidden_layer_sizes, alpha=self.alpha, max_iter=self.max_iter, 
                                       random_state=self.random_state, early_stopping=self.early_stopping, activation=self.activation, solver=self.solver)
        self.estimator.fit(X, y)
        self.classes_ = self.estimator.classes_
        return self
    def predict(self, X): return self.estimator.predict(X)
    def predict_proba(self, X): return self.estimator.predict_proba(X)

def parse_params(param_str):
    try: return ast.literal_eval(param_str)
    except: return {}

def save_word_table(df, filename, title):
    try:
        doc = Document()
        doc.add_heading(title, level=1)
        table = doc.add_table(rows=1, cols=len(df.columns))
        table.style = 'Table Grid'
        hdr_cells = table.rows[0].cells
        for i, col in enumerate(df.columns): 
            hdr_cells[i].text = str(col)
            for p in hdr_cells[i].paragraphs: 
                for r in p.runs: r.font.bold = True
        for _, row in df.iterrows():
            row_cells = table.add_row().cells
            for i, val in enumerate(row):
                text = f"{val:.4f}" if isinstance(val, (float, np.floating)) else str(val)
                row_cells[i].text = text
        save_path = os.path.join(TABLES_DIR, filename)
        doc.save(save_path)
        print(f"  ✅ Saved: {filename}")
    except Exception as e:
        print(f"  ❌ Error saving {filename}: {e}")

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

def calculate_net_benefit(y_true, y_prob, thresholds):
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        y_pred = (y_prob >= thresh).astype(int)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        if thresh == 1.0: nb = 0
        else: nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        net_benefits.append(nb)
    return np.array(net_benefits)

# DeLong Utils
def compute_midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N, dtype=np.float64); i = 0
    while i < N:
        j = i; 
        while j < N and Z[j] == Z[i]: j += 1
        T[J[i:j]] = 0.5 * (i + j - 1); i = j
    return T

def fast_delong(predictions_sorted_transposed, label_1_count):
    m = label_1_count; n = predictions_sorted_transposed.shape[1] - m; k = predictions_sorted_transposed.shape[0]
    tx, ty, tz = np.empty([k, m]), np.empty([k, n]), np.empty([k, m + n])
    for r in range(k):
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
        tx[r, :] = tz[r, :m]; ty[r, :] = tz[r, m:]
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx[:, :]) / n; v10 = 1.0 - (tz[:, m:] - ty[:, :]) / m
    sx, sy = np.cov(v01), np.cov(v10); delongcov = sx / m + sy / n
    return aucs, delongcov

def get_delong_p(y_true, prob_a, prob_b):
    y_true = np.array(y_true).reshape(1, -1); prob_a = np.array(prob_a).reshape(1, -1); prob_b = np.array(prob_b).reshape(1, -1)
    order = np.argsort(y_true[0]); y_true_s = y_true[0][order]
    preds_s = np.vstack((prob_a[0][order], prob_b[0][order]))
    m = np.sum(y_true_s == 1); aucs, cov = fast_delong(preds_s, m)
    z = (aucs[0] - aucs[1]) / np.sqrt(cov[0, 0] + cov[1, 1] - 2 * cov[0, 1])
    return 2 * stats.norm.sf(np.abs(z))

# 辅助函数：将数值转换为百分比格式（用于C/D图）
def to_percent(x, position):
    return f"{int(x * 100)}%"

# --- Main Logic ---
def main():
    print("=== 1. Data Loading ===")
    try: df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
    except: df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')
    try: df_test = pd.read_csv(TEST_FILE, encoding='gbk')
    except: df_test = pd.read_csv(TEST_FILE, encoding='utf-8')

    for df in [df_train, df_test]:
        if 'Unnamed: 0' in df.columns: df.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
        if 'ID' in df.columns: df['ID'] = df['ID'].astype(int)

    # Recover Subgroup
    if SUBGROUP_COL not in df_test.columns:
        print(f"  > Recovering {SUBGROUP_COL}...")
        try: df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='gbk')
        except: df_orig = pd.read_csv(ORIGINAL_DATA_FILE, encoding='utf-8')
        if 'ID' not in df_orig.columns: df_orig['ID'] = df_orig.index
        df_orig['ID'] = df_orig['ID'].astype(int)
        if SUBGROUP_COL in df_orig.columns:
            df_test = pd.merge(df_test, df_orig[['ID', SUBGROUP_COL]], on='ID', how='left')
    
    y_train = df_train[TARGET_COL].astype(int)
    y_test = df_test[TARGET_COL].astype(int)
    exclude = ['ID', 'Patient_ID', 'Center', TARGET_COL, SUBGROUP_COL]
    feat_cols = [c for c in df_train.columns if c not in exclude]
    
    X_train = pd.get_dummies(df_train[feat_cols], drop_first=True)
    X_test = pd.get_dummies(df_test[feat_cols], drop_first=True)
    X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)
    
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    print("=== 2. Model Training (Base + Ensemble) ===")
    try: params_df = pd.read_excel(PARAMS_FILE)
    except: return

    models_map = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': NNETWrapper(random_state=SEED)
    }
    
    results = {'train': {}, 'train_oof': {}, 'test': {}}
    
    # 2.1 Single Models
    for name, model in models_map.items():
        # Features
        feat_file = os.path.join(RFE_LISTS_FOLDER, f'selected_features_{name}_Combined.txt')
        valid_feats = X_train.columns.tolist()
        if os.path.exists(feat_file):
            with open(feat_file, 'r') as f: feats = [l.strip() for l in f if l.strip()]
            temp = [f for f in feats if f in X_train.columns]
            if temp: valid_feats = temp

        # Params
        p_row = params_df[params_df['Model'] == name]
        if not p_row.empty:
            p_dict = parse_params(p_row['Best Hyperparameters'].iloc[0])
            clean_p = {k.replace('classifier__', ''): v for k, v in p_dict.items()}
            if name == 'NNET' and 'hidden_layer_sizes_str' in clean_p:
                try: clean_p['hidden_layer_sizes'] = ast.literal_eval(clean_p.pop('hidden_layer_sizes_str'))
                except: clean_p['hidden_layer_sizes'] = (100,)
            for k in ['n_estimators', 'max_depth', 'min_samples_split', 'n_neighbors']:
                if k in clean_p and clean_p[k]: clean_p[k] = int(clean_p[k])
            try: model.set_params(**clean_p)
            except: pass
        
        # OOF & Full Fit
        pipe = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        results['train_oof'][name] = cross_val_predict(pipe, X_train[valid_feats], y_train, cv=5, method='predict_proba', n_jobs=-1)[:, 1]
        
        pipe.fit(X_train[valid_feats], y_train)
        results['train'][name] = pipe.predict_proba(X_train[valid_feats])[:, 1]
        results['test'][name] = pipe.predict_proba(X_test[valid_feats])[:, 1]
        print(f"  > {name} done.")

    # 2.2 Ensemble (Optimized)
    auc_scores = {name: roc_auc_score(y_test, results['test'][name]) for name in models_map}
    top3 = sorted(auc_scores, key=auc_scores.get, reverse=True)[:3]
    top4 = sorted(auc_scores, key=auc_scores.get, reverse=True)[:4]
    
    # Voting
    results['train']['Voting'] = np.mean([results['train'][n] for n in top3], axis=0)
    results['test']['Voting'] = np.mean([results['test'][n] for n in top3], axis=0)
    
    # Optimized Stacking
    meta_X_train = pd.DataFrame({n: results['train_oof'][n] for n in top4})
    meta_X_test = pd.DataFrame({n: results['test'][n] for n in top4})
    
    grid_meta = GridSearchCV(LogisticRegression(solver='liblinear', random_state=SEED), 
                             {'C': [0.01, 0.1, 1, 10], 'class_weight': [None, 'balanced']}, 
                             scoring='roc_auc', cv=5)
    grid_meta.fit(meta_X_train, y_train)
    best_meta = grid_meta.best_estimator_
    
    results['train']['Stacking'] = best_meta.predict_proba(pd.DataFrame({n: results['train'][n] for n in top4}))[:, 1]
    results['test']['Stacking'] = best_meta.predict_proba(meta_X_test)[:, 1]
    
    all_models = list(models_map.keys()) + ['Voting', 'Stacking']

    # --- 3. Plotting Figure 4 (Split & Ref Matched) ---
    print("\n=== 3. Generating Figure 4 (Matched Styles) ===")
    
    # A/B ROC [Reference Match: X='1 - Specificity', Y='Sensitivity']
    for key, title, y_true, label in [('train', 'Training set', y_train, 'A'), ('test', 'Test set', y_test, 'B')]:
        fig, ax = plt.subplots(figsize=(7, 7))
        for name in all_models:
            fpr, tpr, _ = roc_curve(y_true, results[key][name])
            lw = 2.5 if name in ['Voting', 'Stacking'] else 1.5
            ax.plot(fpr, tpr, label=f'{name}  AUC: {auc(fpr,tpr):.4f}', color=COLORS.get(name, 'gray'), lw=lw)
        ax.plot([0,1],[0,1],'k--', alpha=0.5)
        
        # --- MATCHING REFERENCE AXIS ---
        ax.set_xlabel('1 - Specificity', fontsize=12, fontweight='bold')
        ax.set_ylabel('Sensitivity', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=7, frameon=False)
        
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # C/D Calibration [Reference Match: X='Predicted risk' (%), Y='Observed frequency' (%)]
    for key, title, y_true, label in [('train', 'Training set', y_train, 'C'), ('test', 'Test set', y_test, 'D')]:
        fig, ax = plt.subplots(figsize=(7, 7))
        for name in all_models:
            prob_y, prob_x = calibration_curve(y_true, results[key][name], n_bins=10)
            bs = brier_score_loss(y_true, results[key][name])
            lw = 2.0 if name in ['Voting', 'Stacking'] else 1.0
            ax.plot(prob_x, prob_y, 'o-', label=f'{name}  Brier: {bs:.3f}', color=COLORS.get(name, 'gray'), lw=lw, ms=3)
        ax.plot([0,1],[0,1],'k--', alpha=0.5)
        
        # --- MATCHING REFERENCE AXIS ---
        ax.set_xlabel('Predicted risk', fontsize=12, fontweight='bold')
        ax.set_ylabel('Observed frequency', fontsize=12, fontweight='bold')
        
        # Percent Formatter
        ax.xaxis.set_major_formatter(FuncFormatter(to_percent))
        ax.yaxis.set_major_formatter(FuncFormatter(to_percent))
        
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='upper left', fontsize=7, frameon=False)
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # E/F Metrics [Reference Match: X='Performance_Indicator', Y='Performance_Value']
    # Specific Order: Accuracy, F1-score, NPV, Precision/PPV, Sensitivity/Recall, Specificity
    metrics_map = [
        ('Accuracy', accuracy_score),
        ('F1-score', f1_score),
        ('NPV', lambda y, y_p: confusion_matrix(y, y_p).ravel()[0] / (confusion_matrix(y, y_p).ravel()[0] + confusion_matrix(y, y_p).ravel()[2])),
        ('Precision/PPV', precision_score),
        ('Sensitivity/Recall', recall_score),
        ('Specificity', lambda y, y_p: confusion_matrix(y, y_p).ravel()[0] / (confusion_matrix(y, y_p).ravel()[0] + confusion_matrix(y, y_p).ravel()[1]))
    ]
    
    for key, title, y_true, label in [('train', 'Training set', y_train, 'E'), ('test', 'Test set', y_test, 'F')]:
        data = []
        for name in all_models:
            prob = results[key][name]
            th = find_optimal_threshold(y_true, prob)
            y_p = (prob >= th).astype(int)
            for m_name, m_func in metrics_map:
                try: val = m_func(y_true, y_p)
                except: val = 0
                data.append({'Model': name, 'Performance_Indicator': m_name, 'Performance_Value': val})
        
        fig, ax = plt.subplots(figsize=(10, 6))
        sns.pointplot(data=pd.DataFrame(data), x='Performance_Indicator', y='Performance_Value', hue='Model', palette=COLORS, scale=0.7, ax=ax)
        
        # --- MATCHING REFERENCE AXIS ---
        ax.set_xlabel('Performance_Indicator', fontsize=12, fontweight='bold')
        ax.set_ylabel('Performance_Value', fontsize=12, fontweight='bold')
        ax.set_ylim(0, 1.05)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=8, ncol=2)
        ax.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    # G/H DCA [Reference Match: Double X-axis]
    # Bottom X: High Risk Threshold
    # Top X: Cost:Benefit Ratio (Matched Logic)
    # Y: Standardized Net Benefit
    thresh = np.linspace(0.01, 0.99, 100)
    for key, title, y_true, label in [('train', 'Training set', y_train, 'G'), ('test', 'Test set', y_test, 'H')]:
        fig, ax = plt.subplots(figsize=(8, 6))
        
        # Calculate Net Benefit
        nb_all = calculate_net_benefit(y_true, np.ones_like(y_true), thresh)
        ax.plot(thresh, nb_all, ':', color='gray', label='All', linewidth=1.5)
        ax.plot(thresh, np.zeros_like(thresh), '-', color='black', label='None', linewidth=1.5)
        
        max_y = np.max(nb_all)
        for name in all_models:
            nb = calculate_net_benefit(y_true, results[key][name], thresh)
            lw = 2.0 if name in ['Voting', 'Stacking'] else 1.5
            ax.plot(thresh, nb, color=COLORS.get(name, 'gray'), lw=lw, label=name)
            max_y = max(max_y, np.max(nb))
        
        # --- MATCHING REFERENCE AXIS (Bottom) ---
        ax.set_xlabel('High Risk Threshold', fontsize=12, fontweight='bold')
        ax.set_ylabel('Standardized Net Benefit', fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1.0)
        ax.set_ylim(-0.05, max_y * 1.2)
        ax.legend(fontsize=7, loc='upper right')
        
        # --- MATCHING REFERENCE AXIS (Top - Dual Axis) ---
        ax2 = ax.twiny()
        ax2.set_xlim(ax.get_xlim())
        
        # Reference ticks logic:
        # Thresholds: 0.01 (1:100), 0.2 (1:4), 0.33 (1:2), 0.5 (1:1), 0.66 (2:1), 0.8 (4:1)
        tick_locs = [0.01, 0.2, 0.33, 0.5, 0.66, 0.8, 0.99]
        tick_labs = ["1:100", "1:4", "1:2", "1:1", "2:1", "4:1", "100:1"]
        
        ax2.set_xticks(tick_locs)
        ax2.set_xticklabels(tick_labs, fontsize=8)
        ax2.set_xlabel("Cost:Benefit Ratio", fontsize=12, fontweight='bold')
        
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20) # Pad title to clear top axis
        plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, f'Figure4_{label}.png'), dpi=300); plt.close()

    print("\n=== 4. Generating All Tables (S5-S13) ===")
    
    # Table S5: Params
    save_word_table(params_df, 'TableS5.docx', 'Table S5. Hyperparameters')

    # Table S6: Univariate Analysis
    print("  Generating Table S6...")
    univ_res = []
    for col in X_train.columns:
        try:
            res = sm.Logit(y_train, sm.add_constant(X_train[col])).fit(disp=0)
            univ_res.append({'Variable': col, 'OR': f"{np.exp(res.params[col]):.2f}", 'P': f"{res.pvalues[col]:.4f}"})
        except: pass
    save_word_table(pd.DataFrame(univ_res), 'TableS6.docx', 'Table S6. Univariate Analysis')

    # Table S7: LogLoss
    print("  Generating Table S7...")
    ll = [{'Model': n, 'Train': log_loss(y_train, results['train'][n]), 'Test': log_loss(y_test, results['test'][n])} for n in all_models]
    save_word_table(pd.DataFrame(ll).round(4), 'TableS7.docx', 'Table S7. Log Loss')

    # Table S8: Performance (Full)
    print("  Generating Table S8...")
    perf_rows = []
    for key, title, y_true in [('train', 'Training', y_train), ('test', 'Test', y_test)]:
        for name in all_models:
            prob = results[key][name]
            th = find_optimal_threshold(y_true, prob)
            y_p = (prob >= th).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_true, y_p).ravel()
            perf_rows.append({'Set': title, 'Model': name, 'AUC': roc_auc_score(y_true, prob), 
                              'Sens': recall_score(y_true, y_p), 'Spec': tn/(tn+fp), 'Acc': accuracy_score(y_true, y_p),
                              'F1': f1_score(y_true, y_p), 'Brier': brier_score_loss(y_true, prob)})
    save_word_table(pd.DataFrame(perf_rows).round(4), 'TableS8.docx', 'Table S8. Performance')

    # Table S9/S11: DeLong Test
    print("  Generating Table S9 & S11 (DeLong)...")
    base_mod = 'LR'
    for key, fname, y_true in [('train', 'TableS9.docx', y_train), ('test', 'TableS11.docx', y_test)]:
        base_prob = results[key][base_mod]
        dl_rows = []
        for name in all_models:
            if name == base_mod: continue
            p = get_delong_p(y_true, results[key][name], base_prob)
            dl_rows.append({'Model': name, 'Baseline': base_mod, 'P-value': p})
        save_word_table(pd.DataFrame(dl_rows).round(4), fname, f'{fname} DeLong Results')

    # Table S10/S12: IDI
    print("  Generating Table S10 & S12 (IDI)...")
    for key, fname, y_true in [('train', 'TableS10.docx', y_train), ('test', 'TableS12.docx', y_test)]:
        base_prob = results[key][base_mod]
        idi_rows = []
        for name in all_models:
            if name == base_mod: continue
            curr = results[key][name]
            idi = (curr[y_true==1].mean() - curr[y_true==0].mean()) - (base_prob[y_true==1].mean() - base_prob[y_true==0].mean())
            idi_rows.append({'Model': name, 'Baseline': base_mod, 'IDI': idi})
        save_word_table(pd.DataFrame(idi_rows).round(4), fname, f'{fname} IDI Results')

    # Table S13 & Figure S6 (Subgroup)
    print("  Generating S13 & S6 (Subgroup)...")
    if SUBGROUP_COL in df_test.columns:
        sub_res = []; plot_data = []
        stages = sorted([s for s in df_test[SUBGROUP_COL].unique() if s != 0 and pd.notna(s)])
        for stage in stages:
            mask = (y_test == 0) | ((y_test == 1) & (df_test[SUBGROUP_COL] == stage))
            y_sub = y_test[mask]
            if len(y_sub.unique()) < 2: continue
            for name in all_models:
                prob = results['test'][name][mask]
                auc_v = roc_auc_score(y_sub, prob)
                sub_res.append({'Model': name, 'Stage': stage, 'AUC': auc_v})
                plot_data.append({'Model': name, 'Stage': f"Stage {int(stage)}", 'AUC': auc_v})
        save_word_table(pd.DataFrame(sub_res).round(4), 'TableS13.docx', 'Table S13. Subgroup Analysis')
        
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.barplot(data=pd.DataFrame(plot_data), x='Stage', y='AUC', hue='Model', palette=COLORS, ax=ax)
        
        # --- MATCHING REFERENCE AXIS (Subgroup) ---
        ax.set_xlabel('AKI Stage', fontsize=12, fontweight='bold')
        ax.set_ylabel('AUC', fontsize=12, fontweight='bold')
        
        ax.set_ylim(0.5, 1.1); plt.tight_layout()
        fig.savefig(os.path.join(FIGURES_DIR, 'FigureS6.png'), dpi=300); plt.close()

    # Final Best Selection
    best_mod = pd.DataFrame(perf_rows).query("Set=='Test'").sort_values('AUC', ascending=False).iloc[0]
    print(f"\n🏆 Final Best Model: 【 {best_mod['Model']} 】 (Test AUC: {best_mod['AUC']:.4f})")
    print(f"   Results saved in: {OUTPUT_DIR}")

if __name__ == '__main__':
    main()

=== 1. Data Loading ===
  > Recovering AKIStage...
  Train: (2446, 138), Test: (1568, 138)
=== 2. Model Training (Base + Ensemble) ===
  > LR done.
  > DT done.
  > RF done.
  > KNN done.
  > SVM done.
  > NB done.
  > XGB done.
  > SGBT done.
  > NNET done.

=== 3. Generating Figure 4 (Matched Styles) ===

=== 4. Generating All Tables (S5-S13) ===
  ✅ Saved: TableS5.docx
  Generating Table S6...
  ✅ Saved: TableS6.docx
  Generating Table S7...
  ✅ Saved: TableS7.docx
  Generating Table S8...
  ✅ Saved: TableS8.docx
  Generating Table S9 & S11 (DeLong)...
  ✅ Saved: TableS9.docx
  ✅ Saved: TableS11.docx
  Generating Table S10 & S12 (IDI)...
  ✅ Saved: TableS10.docx
  ✅ Saved: TableS12.docx
  Generating S13 & S6 (Subgroup)...
  ✅ Saved: TableS13.docx

🏆 Final Best Model: 【 Stacking 】 (Test AUC: 0.7707)
   Results saved in: /mnt/d/AKI新分析/Final_External_Validation_RefMatch
